Read in, merge, and clean the four datasets to make a single combined pandas DataFrame. 

The DataFram whould be  called employees_final with the following columns, in the exact order listed: employee_first_name, employee_last_name, employee_country, employee_city, employee_street, employee_street_number, emergency_contact, emergency_contact_number, relationship, monthly_salary, team, title, office, office_country, office_city, office_street, office_street_number.



In [46]:
# Import pandas
import pandas as pd

# Import the office addresses csv dataset
office_address = pd.read_csv("datasets/office_addresses.csv")
print(office_address.head())

          office office_country    office_city   office_street  \
0  Leuven Office             BE         Leuven  Martelarenlaan   
1     ESB Office             US  New York City    Fifth Avenue   
2  WeWork Office             GB         London      Old Street   

   office_street_number  
0                    38  
1                   350  
2                   207  


In [47]:
# Import the employee addresses excel dataset
employee_address = pd.read_excel("datasets/employee_information.xlsx", sheet_name="employee_addresses")

# Set employee_id as the index and rearrange columns for merging
employee_address.set_index("employee_id", inplace=True)
merge_address = employee_address[["employee_first_name", "employee_last_name", "employee_country", "employee_city", "employee_street", "employee_street_number"]]
print(merge_address.head())

            employee_first_name employee_last_name employee_country  \
employee_id                                                           
A2R5H9                      Jax             Hunman               BE   
H8K0L6                     Tara               Siff               GB   
G4R7V0                    Gemma              Sagal               US   
M1Z7U9                      Tig             Coates               FR   

            employee_city      employee_street  employee_street_number  
employee_id                                                             
A2R5H9             Leuven          Grote Markt                       9  
H8K0L6             London         Baker Street                     221  
G4R7V0           New-York         Perry Street                      66  
M1Z7U9              Paris  Rue de l'Université                       7  


In [48]:
# Import the emergency contact excel dataset and add the headers
emergency_contact = pd.read_excel("datasets/employee_information.xlsx", sheet_name= "emergency_contacts", header=None, names = ["employee_id", "last_name", "first_name", "emergency_contact", "emergency_contact_number", "relationship"])

# Set employee_id as the index and rearrange columns for merging
emergency_contact.set_index("employee_id", inplace=True)
merge_contact= emergency_contact[["emergency_contact", "emergency_contact_number", "relationship"]]
print(merge_contact.head())

            emergency_contact emergency_contact_number relationship
employee_id                                                        
A2R5H9             Opie Hurst          +32-456-5556-84      Brother
H8K0L6        Wendy de Matteo         +44-020-5554-333       Sister
G4R7V0           John Newmark           +1-202-555-194      Husband
M1Z7U9            Venus Noone          +1-202-555-0130         Wife


In [49]:
# Import the employee roles JSON dataset
employee_role = pd.read_json("datasets/employee_roles.json", orient = "index")

# Name the index and rearrange columns for merging
employee_role.index.name = "employee_id"
merge_role = employee_role[["monthly_salary", "team", "title"]]
print(merge_role.head())

            monthly_salary               team               title
employee_id                                                      
A2R5H9               $4500         Leadership                 CEO
H8K0L6               $4500         Leadership                 CFO
G4R7V0               $3000              Sales  Business Developer
M1Z7U9               $2000  People Operations      Office Manager


In [50]:
# Join the 3 dataframes with employee_id as the index
employee_joined = merge_address.join([merge_contact, merge_role])

# Merge with the office addresses dataframe on office_country/country and reset the index to office_id
employees_final = pd.merge(employee_joined.reset_index(), office_address, left_on ="employee_country", right_on = "office_country", how= "left").set_index("employee_id")

# Filter for columns beginning with "office" and fill NA with "remote"
office_columns = employees_final.filter(regex="^office", axis =1).columns
employees_final[office_columns] = employees_final[office_columns].fillna("Remote")
print(employees_final)


            employee_first_name employee_last_name employee_country  \
employee_id                                                           
A2R5H9                      Jax             Hunman               BE   
H8K0L6                     Tara               Siff               GB   
G4R7V0                    Gemma              Sagal               US   
M1Z7U9                      Tig             Coates               FR   

            employee_city      employee_street  employee_street_number  \
employee_id                                                              
A2R5H9             Leuven          Grote Markt                       9   
H8K0L6             London         Baker Street                     221   
G4R7V0           New-York         Perry Street                      66   
M1Z7U9              Paris  Rue de l'Université                       7   

            emergency_contact emergency_contact_number relationship  \
employee_id                                              